# 第2章：实践方法论

> "模型训练不顺利，不要盲目调参——先诊断问题，再对症下药。"

## 本章知识导图

```
实践方法论——训练不顺时的系统诊断框架
│
├── 先看训练集损失
│   ├── 训练损失很大？
│   │   ├── 2.1 模型偏差：模型太简单，最优参数也拟合不了
│   │   │   └── 解法：增加层数/神经元/特征，换更灵活的模型
│   │   └── 2.2 优化问题：模型够强但梯度下降没找到最优解
│   │       ├── 诊断技巧：深模型 vs 浅模型比较法
│   │       └── 解法：换优化器(Adam)、调学习率、动量、更好的初始化
│   └── 训练损失小 → 看测试集损失
│       ├── 测试损失大？
│       │   ├── 2.3 过拟合：模型记住了训练数据的噪声
│       │   │   └── 解法：正则化(L1/L2)、数据增强、早停、Dropout、减少参数
│       │   └── 2.5 不匹配：训练和测试数据来自不同分布
│       │       └── 解法：收集更多匹配的训练数据、迁移学习
│       └── 2.4 交叉验证：如何公平评估模型的泛化能力
└── 总结：先诊断 → 再开药 → 不要瞎调参！
```

## 2.1 模型偏差 (Model Bias)

### 什么是模型偏差？

**模型池太浅——所有候选函数中，最好的那个也拟合不了数据。**

举个例子：数据是二次曲线$y=x^2$，但你只给了线性模型$y=b+wx$。不管怎么调$(b,w)$，直线永远不可能变成抛物线——这就是模型偏差。

### 如何识别？

- **症状：** 训练损失很大，而且无论怎么调参都降不下去
- **判断方法：** 尝试增大模型（更多层、更多神经元）。如果训练损失明显下降 → 说明之前是模型偏差

### 解决方案

1. **增加模型灵活性**：更多层、更多神经元、更复杂的激活函数
2. **增加特征**：输入更多相关信息（如预测观看次数时，从1天增到7天、28天）
3. **换架构**：图像问题从全连接换成CNN，序列问题从RNN换成Transformer

> **关键认知：** 模型偏差不是优化的问题——就算给你全局最优参数，模型本身也拟合不了数据。唯一的解法是**增大模型**。

## 2.2 优化问题 (Optimization Issue)

### 什么是优化问题？

**模型池够深——包含真正能拟合数据的函数——但梯度下降没能找到那个函数。**

这就像一个山谷，你站在山顶但看不见全局。你跟着坡度往下走，却走进了一个小坑（局部最小值/鞍点），真正的谷底在别处。

### 关键诊断技巧：深模型 vs 浅模型比较

这是李宏毅老师给出的**最实用的诊断技巧**：

1. 训练一个浅模型（如20层ResNet），记录训练损失$L_{\text{shallow}}$
2. 训练一个深模型（如50层ResNet），记录训练损失$L_{\text{deep}}$
3. **如果$L_{\text{deep}} > L_{\text{shallow}}$ → 一定是优化问题！**

为什么？因为深模型**可以退化成浅模型**（把多余的层设成恒等映射即可）。如果深模型的损失反而更高，只能说明**梯度下降没能找到深模型的最优解**，而不是深模型"太复杂"。

### 解决方案

- 换更好的优化器（Adam替代SGD）
- 调整学习率（可能需要更大或更小的学习率）
- 使用动量法
- 更好的参数初始化（如He初始化、Xavier初始化）
- 学习率预热(Warmup)

## 2.3 过拟合 (Overfitting)

### 定义

$$\text{过拟合} \iff \text{训练损失小} \land \text{测试损失大}$$

模型把训练数据中的噪声和随机波动也"死记硬背"下来了，导致遇到新数据时表现很差。

### 为什么会过拟合？

- **模型太灵活**（参数太多） + **数据太少** → 模型能记住每个训练样本，却无法泛化
- 类比：考试前把所有习题的答案背下来，但考试换了新题就不会做

### 解决方法详解

| 方法 | 原理 | 实现 |
|------|------|------|
| **L2正则化** | 在损失中加$\lambda\|\mathbf{w}\|^2$，惩罚大权重 | `optimizer(..., weight_decay=1e-4)` |
| **L1正则化** | 加$\lambda\|\mathbf{w}\|_1$，产生稀疏权重 | 需手动实现 |
| **Dropout** | 训练时随机丢弃神经元，相当于训练多个子网络的集成 | `nn.Dropout(p=0.5)` |
| **早停(Early Stopping)** | 验证损失不再下降时停止训练 | 监控`val_loss` |
| **数据增强(Data Augmentation)** | 对训练数据做随机变换增加多样性 | 翻转、旋转、裁剪、颜色抖动 |
| **减少模型参数** | 降低模型灵活性 | 更少的层/神经元 |

> **正则化的本质：** 在"拟合训练数据"和"保持简单"之间做权衡。$\lambda$越大 → 模型越简单 → 越不容易过拟合，但可能欠拟合（模型偏差）。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 演示过拟合与Dropout的效果
def make_data(n=30):
    x = torch.linspace(-3, 3, n).reshape(-1, 1)
    y = x**2 + torch.randn(n, 1) * 0.5
    return x, y

x_train, y_train = make_data(30)
x_test, y_test = make_data(50)

# 过于复杂的模型（300个ReLU，30个数据点 → 必过拟合）
big_model = nn.Sequential(nn.Linear(1, 200), nn.ReLU(), nn.Linear(200, 200), nn.ReLU(), nn.Linear(200, 1))
big_model_dropout = nn.Sequential(nn.Linear(1, 200), nn.ReLU(), nn.Dropout(0.5), nn.Linear(200, 200), nn.ReLU(), nn.Dropout(0.5), nn.Linear(200, 1))

def train(m, name, epochs=3000):
    opt = torch.optim.Adam(m.parameters(), lr=0.005)
    m.train()
    train_losses, test_losses = [], []
    for e in range(epochs):
        opt.zero_grad()
        loss = nn.MSELoss()(m(x_train), y_train)
        loss.backward()
        opt.step()
        if e % 200 == 0:
            m.eval()
            train_losses.append(loss.item())
            test_losses.append(nn.MSELoss()(m(x_test), y_test).item())
            m.train()
    return train_losses, test_losses

print("训练大型网络（30个数据点）...")
loss_tr1, loss_te1 = train(big_model, "无Dropout")
loss_tr2, loss_te2 = train(big_model_dropout, "有Dropout")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(loss_tr1, label="Train(无Dropout)", marker='o')
axes[0].plot(loss_te1, label="Test(无Dropout)", marker='s')
axes[0].set_title("无Dropout → 明显过拟合"); axes[0].legend()
axes[1].plot(loss_tr2, label="Train(有Dropout)", marker='o')
axes[1].plot(loss_te2, label="Test(有Dropout)", marker='s')
axes[1].set_title("有Dropout → 过拟合减轻"); axes[1].legend()

# 画拟合曲线
big_model.eval(); big_model_dropout.eval()
xp = torch.linspace(-4, 4, 200).reshape(-1, 1)
axes[2].scatter(x_train, y_train, s=15, label="Training data")
axes[2].plot(xp, big_model(xp).detach(), alpha=0.7, label="无Dropout(过度扭曲)")
axes[2].plot(xp, big_model_dropout(xp).detach(), alpha=0.7, label="有Dropout(更平滑)")
axes[2].plot(xp, xp**2, 'k--', alpha=0.5, label="真实函数: y=x^2")
axes[2].legend(); axes[2].set_ylim(-2, 12)
plt.tight_layout(); plt.show()
print("无Dropout的训练损失极小但测试损失大 —— 过拟合")
print("有Dropout后限制了模型复杂度，泛化更好")

## 2.4 交叉验证 (Cross Validation)

### 为什么需要交叉验证？

单次划分训练/验证集有运气成分——可能恰好分到"简单"的验证集，高估了模型性能。

K折交叉验证消除这种随机性。

### K折交叉验证流程

1. 将训练集分成K等份
2. 逐次用K-1份做训练，1份做验证
3. 重复K次，每次换不同的一份做验证
4. 取K次验证分数的平均值

### 黄金法则

| 数据划分 | 用途 |
|---------|------|
| 训练集(Train) | 训练模型参数 |
| 验证集(Validation) | 调超参数、选模型、Early Stopping |
| 测试集(Test) | **只用一次**——最终报告性能 |

> ❌ **绝对禁止：** 在测试集上调超参数！这等于用"答案"去优化"解题方法"——你看到的测试分数已经不可信了。

## 2.5 不匹配 (Mismatch)

### 不匹配 vs 过拟合

| 维度 | 过拟合 | 不匹配 |
|------|--------|--------|
| 原因 | 模型太复杂，记住了噪声 | 训练和测试数据来自**不同分布** |
| 症状 | 训练损失小，验证损失大 | 训练/验证损失小，测试损失大 |
| 解法 | 正则化、Dropout、数据增强 | 收集更多匹配数据、迁移学习 |
| 关键区别 | 训练/验证同分布 | 训练/验证不同分布（或验证/测试不同分布） |

### 不匹配的例子
- 训练数据是高清正面照，测试是监控摄像头侧脸
- 训练是标准普通话，用户说方言
- 训练数据来自2017-2020，预测2021的观看习惯已改变

> **关键：** 不匹配时增加正则化**反而可能更差**！因为问题不在模型复杂度，而在数据分布差异。

## 诊断决策树（全书核心方法论）

```
训练损失大？
  ├─ Yes → 1) 换更大模型 → 损失降了？→ 之前的模型有偏差
  │               → 损失没降？→ 优化问题（换优化器/调学习率）
  └─ No  → 测试损失大？
            ├─ Yes → 训练/验证/测试都是同分布？
            │         ├─ Yes → 过拟合 → 正则化/数据增强/早停
            │         └─ No  → 不匹配 → 收集匹配数据
            └─ No  → 恭喜！模型训练成功！
```

## 本章核心收获

1. **先诊断，再开药**：盲目调参是最低效的做法
2. **模型偏差**：训练损失大 → 增大模型
3. **优化问题**：深模型比浅模型差 → 优化器有问题
4. **过拟合**：训练好测试差 → 正则化/数据增强/早停/Dropout
5. **不匹配**：数据分布不同 → 收集匹配数据，正则化没用
6. **交叉验证**：公平评估模型，绝对不要在测试集上调参
7. 本章的"诊断→解决"思维是贯穿全书的方法论核心